# Ejercicio 7: Bases de Datos Vectoriales

## Objetivo de la práctica

Entender el concepto de Bases de Datos Vectoriales y saber utilizar las herramientas actuales

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus


In [5]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [6]:
# Set the path to the file you'd like to load
file_path = "wikipedia_text_corpus.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects",
  file_path,
)

df.head()

,Unnamed: 0,text
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...
1,2,Battery indicator\n\nA battery indicator (also...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...


## Parte 1: Generación de Embeddings

Vamos a utilizar E5 como modelo de embeddings.

La documentación de E5 está disponible desde este [link](https://huggingface.co/intfloat/e5-base-v2)

### Actividad

1. Normalizar el corpus
2. Definir una función `chunk_text`, y dividir los textos en _chunks_.
3. Generar embeddings por cada _chunk_

In [7]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import re

df = df.dropna(subset=["text"]).reset_index(drop=True)

# subconjunto representativo para viabilizar ejecucion en CPU
df = df.sample(n=500, random_state=42).reset_index(drop=True)

def normalize_text(s):
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text_norm"] = df["text"].astype(str).map(normalize_text)
print(f"Documentos: {len(df)}")
df.head()

Documentos: 500


,Unnamed: 0,text,text_norm
0,34,Rodrig Goliescu\n\nRodrig Goliescu (1882â€“194...,Rodrig Goliescu Rodrig Goliescu (1882â€“1942) ...
1,7235,Push-button\n\nA push-button (also spelled pus...,Push-button A push-button (also spelled pushbu...
2,8682,WriteOnline\n\nWriteOnline is an online word p...,WriteOnline WriteOnline is an online word proc...
3,5983,Four factor formula\n\nThe four-factor formula...,"Four factor formula The four-factor formula, a..."
4,7066,Delphi (online service)\n\nDelphi Forums is a ...,Delphi (online service) Delphi Forums is a U.S...


In [8]:
def chunk_text(text, max_chars=800, overlap=100):
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + max_chars, n)
        chunk = text[start:end].strip()
        if len(chunk) > 0:
            chunks.append(chunk)
        if end == n:
            break
        start = max(0, end - overlap)
    return chunks

records = []
for i, row in df.iterrows():
    for j, ch in enumerate(chunk_text(row["text_norm"])):
        records.append({"doc_id": int(i), "chunk_id": j, "text": ch})

chunks_df = pd.DataFrame(records)
print(f"Total chunks: {len(chunks_df)}")
chunks_df.head()

Total chunks: 3773


,doc_id,chunk_id,text
0,0,0,Rodrig Goliescu Rodrig Goliescu (1882â€“1942) ...
1,0,1,"he sent a survey ""Laws of air dynamics"" to the..."
2,0,2,eer Luigi Stipa will build an aircraft with a ...
3,1,0,Push-button A push-button (also spelled pushbu...
4,1,1,"appliances, and various other mechanical and e..."


In [9]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    chunks_df["text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

print(embeddings.shape, embeddings.dtype)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/59 [00:00<?, ?it/s]

(3773, 384) float32


Función para codificar queries y una consulta de prueba

In [10]:
def embed_query(text):
    vec = model.encode([text], convert_to_numpy=True, normalize_embeddings=True)
    return vec.astype("float32")

query_text = "Battery measuring"
query_vec = embed_query(query_text)
print(query_vec.shape)

(1, 384)


Se guarda todo 

In [11]:
np.save("embeddings_minil.npy", embeddings)
chunks_df.to_parquet("chunks_df.parquet", index=False)
print("guardado OK")

guardado OK


## Parte 2: FAISS

FAISS es una librería para búsqueda por similitud eficiente y clustering de vectores densos.

La documentación de FAISS está disponible en este [link](https://faiss.ai/index.html)

### Actividad

1. Crea un índice en FAISS
2. Carga los embeddings
3. Realiza una búsqueda a partir de una _query_

In [12]:
import faiss

dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)
print(f"Vectores indexados: {index.ntotal}")

k = 10
D, I = index.search(query_vec, k)

print(f"\nQuery: '{query_text}'\n")
for rank, (idx, dist) in enumerate(zip(I[0], D[0])):
    texto = chunks_df.iloc[idx]["text"][:120]
    print(f"  [{rank+1}] id={idx}  dist={dist:.4f}  → {texto}...")

Vectores indexados: 3773

Query: 'Battery measuring'

  [1] id=2185  dist=1.2423  → ent on the partial pressure of oxygen in the gas to which the cell is exposed: Linearity is achieved by placing a diffus...
  [2] id=122  dist=1.2536  → r cooled to store or extract energy. However, in a non-phase change encapsulated thermal battery the temperature of the ...
  [3] id=117  dist=1.2830  → Thermal Battery A thermal energy battery is a physical structure used for the purpose of storing and releasing thermal e...
  [4] id=3033  dist=1.2961  → sible without outshining the target. The reticle has markings that match targets of various heights from 0.3Â m to 2.7Â ...
  [5] id=2183  dist=1.3490  → Electro-galvanic oxygen sensor An electro-galvanic fuel cell is an electrochemical device which consumes a fuel to produ...
  [6] id=2099  dist=1.3505  → y made than D â€“ A. The result is that an electric current can be drawn through the molecule if the electrons are added...
  [7] id=2949  dist=1.35

Ejecutar en terminal, asegurarse de que docker este instalado.

_docker run -d --name qdrant -p 6333:6333 qdrant/qdrant_

## Parte 3 — Vector DB #1: Qdrant (búsqueda vectorial + metadata)

### Objetivo
Recrear el mismo flujo que con FAISS, pero usando una base vectorial con soporte nativo de **metadata** y filtros.

### Qué debes implementar
1. Levantar / conectar con una instancia de Qdrant.
2. Crear una colección con:
   - dimensión `D` (la de tus embeddings)
   - métrica (cosine o L2)
3. Insertar:
   - `id`
   - `embedding`
   - `payload` (metadata: texto, título, etiquetas, etc.)
4. Consultar Top-k por similitud:
   - `query_embedding`
   - `k`

### Inputs esperados (ya definidos arriba en el notebook)
- `embeddings`: matriz `N x D` (float32)
- `texts`: lista de `N` strings
- `metadatas`: lista de `N` dicts (opcional)
- `query_text`: string
- `query_embedding`: vector `1 x D`

### Entregable
- Una función `qdrant_search(query_embedding, k)` que retorne:
  - lista de `(id, score, text, metadata)`
- Un ejemplo de consulta con `k=5` y su salida.

### Preguntas
- ¿La métrica usada fue cosine o L2? ¿Por qué?
- ¿Qué tan fácil fue filtrar por metadata en comparación con FAISS?
- ¿Qué pasa con el tiempo de respuesta cuando aumentas `k`?


1. Levantar / conectar con una instancia de Qdrant.

In [13]:
!pip install qdrant-client -q

In [14]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

client = QdrantClient(host="localhost", port=6333)

2. Crear una colección con:
   - dimensión `D` (la de tus embeddings)
   - métrica (cosine o L2)

In [15]:
client.recreate_collection(
    collection_name="wiki_chunks",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)

C:\Users\ladol\AppData\Local\Temp\ipykernel_18588\2884048361.py:1: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

3. Insertar:
   - `id`
   - `embedding`
   - `payload` (metadata: texto, título, etiquetas, etc.)

In [16]:
points = []
for i, row in chunks_df.iterrows():
    points.append(PointStruct(
        id=i,
        vector=embeddings[i].tolist(),
        payload={"text": row["text"][:500], "doc_id": row["doc_id"], "chunk_id": row["chunk_id"]}
    ))

In [ ]:
for start in range(0, len(points), 500):
    client.upsert("wiki_chunks", points[start:start+500])

print(f"Insertados: {client.get_collection('wiki_chunks').points_count}")

Insertados: 3773


4. Consultar Top-k por similitud:
   - `query_embedding`
   - `k`

In [18]:
def qdrant_search(query_embedding, k=5):
    results = client.query_points(
        collection_name="wiki_chunks",
        query=query_embedding[0].tolist(),
        limit=k
    ).points
    salida = []
    for r in results:
        salida.append((r.id, r.score, r.payload["text"][:120], r.payload))
    return salida

for id_, score, texto, meta in qdrant_search(query_vec, k=5):
    print(f"  id={id_}  score={score:.4f}  → {texto}...")

  id=2185  score=0.3788  → ent on the partial pressure of oxygen in the gas to which the cell is exposed: Linearity is achieved by placing a diffus...
  id=122  score=0.3732  → r cooled to store or extract energy. However, in a non-phase change encapsulated thermal battery the temperature of the ...
  id=117  score=0.3585  → Thermal Battery A thermal energy battery is a physical structure used for the purpose of storing and releasing thermal e...
  id=3033  score=0.3519  → sible without outshining the target. The reticle has markings that match targets of various heights from 0.3Â m to 2.7Â ...
  id=2183  score=0.3255  → Electro-galvanic oxygen sensor An electro-galvanic fuel cell is an electrochemical device which consumes a fuel to produ...


### Respuestas

- **¿Cosine o L2?** Se usó cosine porque los embeddings ya están normalizados. Con vectores normalizados cosine es más interpretable: 1 = idéntico, 0 = sin relación.

- **¿Filtrado por metadata vs FAISS?** Mucho más fácil en Qdrant. FAISS no tiene soporte nativo de metadata — hay que filtrar manualmente después de la búsqueda. Qdrant permite filtrar directamente en la query con `payload`.

- **¿Tiempo de respuesta al aumentar k?** En un corpus de este tamaño el cambio es mínimo. En corpora grandes el aumento sería más notorio porque debe ordenar más candidatos.

## Parte 4 — Vector DB #2: Milvus (indexación ANN y escalabilidad)

### Objetivo
Implementar el flujo de indexación + búsqueda con una base vectorial orientada a escalabilidad.

### Qué debes implementar
1. Conectar a Milvus.
2. Crear un esquema (colección) con:
   - campo `id` (entero o string)
   - campo `embedding` (vector `D`)
   - campos de metadata (p.ej., `category`, `source`, `title`)
3. Insertar `N` embeddings.
4. Crear/seleccionar un índice ANN (ej. HNSW o IVF).
5. Ejecutar consultas Top-k y recuperar textos asociados.

### Recomendación didáctica
Haz dos configuraciones:
- **Búsqueda exacta** (si aplica) o configuración “más precisa”
- **Búsqueda ANN** (configuración “más rápida”)

Luego compara:
- tiempo de consulta
- overlap de resultados (cuántos IDs coinciden)

### Entregable
- Función `milvus_search(query_embedding, k)` que devuelva resultados.
- Un mini experimento: `k=5` y `k=20` (tiempos y resultados).

### Preguntas
- ¿Qué parámetros del índice/control de búsqueda ajustaste para precisión vs velocidad?
- ¿Qué evidencia tienes de que ANN cambia los resultados (aunque sea poco)?


In [19]:
!pip install milvus-lite -q

1. Conectar a Milvus.

In [23]:
from pymilvus import MilvusClient, DataType

mclient = MilvusClient("milvus_local.db")
print("Conectado OK")

Conectado OK


2. Crear un esquema (colección) con:
   - campo `id` (entero o string)
   - campo `embedding` (vector `D`)
   - campos de metadata (p.ej., `category`, `source`, `title`)

In [24]:
schema = mclient.create_schema(auto_id=False)
schema.add_field("id", DataType.INT64, is_primary=True)
schema.add_field("vector", DataType.FLOAT_VECTOR, dim=384)
schema.add_field("text", DataType.VARCHAR, max_length=600)
schema.add_field("doc_id", DataType.INT64)
schema.add_field("chunk_id", DataType.INT64)

mclient.create_collection(
    collection_name="wiki_chunks",
    schema=schema
)

3. Insertar `N` embeddings.


In [25]:
data = []
for i, row in chunks_df.iterrows():
    data.append({
        "id": i,
        "vector": embeddings[i].tolist(),
        "text": row["text"][:500],
        "doc_id": int(row["doc_id"]),
        "chunk_id": int(row["chunk_id"])
    })

for start in range(0, len(data), 500):
    mclient.insert("wiki_chunks", data[start:start+500])

print(f"Insertados: {mclient.get_collection_stats('wiki_chunks')}")

Insertados: {'row_count': 3773}


4. Crear/seleccionar un índice ANN (ej. HNSW o IVF).


In [26]:
index_params = mclient.prepare_index_params()
index_params.add_index(
    field_name="vector",
    index_type="HNSW",
    metric_type="COSINE",
    params={"M": 16, "efConstruction": 200}
)
mclient.create_index("wiki_chunks", index_params)

5. Ejecutar consultas Top-k y recuperar textos asociados.

In [27]:
import time

def milvus_search(query_embedding, k=5):
    results = mclient.search(
        collection_name="wiki_chunks",
        data=[query_embedding[0].tolist()],
        limit=k,
        output_fields=["text", "doc_id", "chunk_id"]
    )
    salida = []
    for hit in results[0]:
        salida.append((hit["id"], hit["distance"], hit["entity"]["text"][:120]))
    return salida

In [28]:
print("Top 5:")
for id_, score, texto in milvus_search(query_vec, k=5):
    print(f"  id={id_}  score={score:.4f}  → {texto}...")

Top 5:
  id=2185  score=0.6212  → ent on the partial pressure of oxygen in the gas to which the cell is exposed: Linearity is achieved by placing a diffus...
  id=122  score=0.6268  → r cooled to store or extract energy. However, in a non-phase change encapsulated thermal battery the temperature of the ...
  id=117  score=0.6415  → Thermal Battery A thermal energy battery is a physical structure used for the purpose of storing and releasing thermal e...
  id=3033  score=0.6481  → sible without outshining the target. The reticle has markings that match targets of various heights from 0.3Â m to 2.7Â ...
  id=2183  score=0.6745  → Electro-galvanic oxygen sensor An electro-galvanic fuel cell is an electrochemical device which consumes a fuel to produ...


In [29]:
print("\nTop 20:")
for id_, score, texto in milvus_search(query_vec, k=20):
    print(f"  id={id_}  score={score:.4f}  → {texto}...")


Top 20:
  id=2185  score=0.6212  → ent on the partial pressure of oxygen in the gas to which the cell is exposed: Linearity is achieved by placing a diffus...
  id=122  score=0.6268  → r cooled to store or extract energy. However, in a non-phase change encapsulated thermal battery the temperature of the ...
  id=117  score=0.6415  → Thermal Battery A thermal energy battery is a physical structure used for the purpose of storing and releasing thermal e...
  id=3033  score=0.6481  → sible without outshining the target. The reticle has markings that match targets of various heights from 0.3Â m to 2.7Â ...
  id=2183  score=0.6745  → Electro-galvanic oxygen sensor An electro-galvanic fuel cell is an electrochemical device which consumes a fuel to produ...
  id=2099  score=0.6753  → y made than D â€“ A. The result is that an electric current can be drawn through the molecule if the electrons are added...
  id=2949  score=0.6761  → ed areas are thick copper busbars of almost zero resistance.

In [ ]:
# comparacion de tiempos
t5 = time.time(); milvus_search(query_vec, k=5); t5 = time.time() - t5
t20 = time.time(); milvus_search(query_vec, k=20); t20 = time.time() - t20
print(f"\nTiempo k=5: {t5:.4f}s  |  k=20: {t20:.4f}s")


Tiempo k=5: 0.9451s  |  k=20: 0.9569s


### Respuestas

- **¿Qué parámetros ajustaste?** En HNSW se configuró M=16 (conexiones por nodo) y efConstruction=200 (profundidad al construir el grafo). Mayor M y efConstruction dan más precisión pero más tiempo de construcción.

- **¿Evidencia de que ANN cambia resultados?** Con un corpus de 3773 vectores la diferencia es imperceptible, HNSW encuentra los mismos resultados que la búsqueda exacta de FAISS. En corpora de millones de vectores, ANN sacrificaría algunos vecinos reales a cambio de velocidad.

## Parte 5 — Vector DB #3: Weaviate (búsqueda semántica con esquema)

### Objetivo
Montar una colección con esquema (clase) y ejecutar búsquedas semánticas Top-k, opcionalmente con filtros.

### Qué debes implementar
1. Conectar a Weaviate.
2. Definir un esquema:
   - Clase/colección (por ejemplo `Document`)
   - Propiedades: `text`, `title`, `category`, etc.
   - Vector asociado (embedding)
3. Insertar objetos con:
   - propiedades + vector
4. Consultar por similitud (Top-k) con `query_embedding`.
5. (Opcional) agregar un filtro por propiedad (metadata).

### Recomendación
Asegúrate de guardar el `text` original y al menos 1 campo de metadata para probar filtrado.

### Entregable
- Función `weaviate_search(query_embedding, k)` que retorne:
  - id, score, text, metadata

### Preguntas
- ¿Qué diferencia conceptual encuentras entre “schema + objetos” vs “tabla + filas”?
- ¿Cómo describirías el trade-off de complejidad vs expresividad?


docker run -d --name weaviate -p 8080:8080 -p 50051:50051 -e AUTHENTICATION_ANONYMOUS_ACCESS_ENABLED=true -e PERSISTENCE_DATA_PATH=/var/lib/weaviate cr.weaviate.io/semitechnologies/weaviate:latest

In [31]:
!pip install weaviate-client -q

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.6 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 6.33.6 which is incompatible.
pyopenssl 24.2.1 requires cryptography<44,>=41.0.5, but you have cryptography 49.0.0 which is incompatible.
streamlit 1.37.1 requires packaging<25,>=20, but you have packaging 25.0 which is incompatible.
streamlit 1.37.1 requires pillow<11,>=7.1.0, but you have pillow 11.3.0 which is incompatible.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 6.33.6 which is incompatible.
streamlit 1.37.1 requires rich<14,>=10.14.0, but you have rich 15.0.0 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.

1. Conectar a Weaviate.

In [32]:
import weaviate

wclient = weaviate.connect_to_local(host="localhost", port=8081, grpc_port=50051)
print("Conectado:", wclient.is_ready())

Conectado: True


2. Definir un esquema:
   - Clase/colección (por ejemplo `Document`)
   - Propiedades: `text`, `title`, `category`, etc.
   - Vector asociado (embedding)

In [ ]:
from weaviate.classes.config import Property, DataType, Configure

wclient.collections.delete("WikiChunk")
wclient.collections.create(
    name="WikiChunk",
    properties=[
        Property(name="text", data_type=DataType.TEXT),
        Property(name="doc_id", data_type=DataType.INT),
        Property(name="chunk_id", data_type=DataType.INT),
    ],
    vectorizer_config=Configure.Vectorizer.none()  # usamos nuestros propios embeddings
)

print("Coleccion WikiChunk creada")

Coleccion WikiChunk creada


c:\Users\ladol\anaconda3\Lib\site-packages\weaviate\warnings.py:196: DeprecationWarning: Dep024: You are using the `vectorizer_config` argument in `collection.config.create()`, which is deprecated.
            Use the `vector_config` argument instead.
            
  warnings.warn(


3. Insertar objetos con:
   - propiedades + vector

In [34]:
collection = wclient.collections.get("WikiChunk")

with collection.batch.dynamic() as batch:
    for i, row in chunks_df.iterrows():
        batch.add_object(
            properties={
                "text": row["text"][:500],
                "doc_id": int(row["doc_id"]),
                "chunk_id": int(row["chunk_id"])
            },
            vector=embeddings[i].tolist()
        )

print(f"Insertados: {collection.aggregate.over_all().total_count}")

Insertados: 3773


4. Consultar por similitud (Top-k) con `query_embedding`.

In [35]:
from weaviate.classes.query import MetadataQuery

def weaviate_search(query_embedding, k=5):
    results = collection.query.near_vector(
        near_vector=query_embedding[0].tolist(),
        limit=k,
        return_metadata=MetadataQuery(distance=True),
        return_properties=["text", "doc_id", "chunk_id"]
    )
    salida = []
    for obj in results.objects:
        salida.append((
            str(obj.uuid)[:8],
            obj.metadata.distance,
            obj.properties["text"][:120],
            obj.properties
        ))
    return salida

for id_, score, texto, meta in weaviate_search(query_vec, k=5):
    print(f"  id={id_}  dist={score:.4f}  → {texto}...")

  id=c1b7029e  dist=0.6212  → ent on the partial pressure of oxygen in the gas to which the cell is exposed: Linearity is achieved by placing a diffus...
  id=6d538216  dist=0.6268  → r cooled to store or extract energy. However, in a non-phase change encapsulated thermal battery the temperature of the ...
  id=93086ae2  dist=0.6415  → Thermal Battery A thermal energy battery is a physical structure used for the purpose of storing and releasing thermal e...
  id=7fb42ddf  dist=0.6481  → sible without outshining the target. The reticle has markings that match targets of various heights from 0.3Â m to 2.7Â ...
  id=7da85e9e  dist=0.6745  → Electro-galvanic oxygen sensor An electro-galvanic fuel cell is an electrochemical device which consumes a fuel to produ...


5. (Opcional) agregar un filtro por propiedad (metadata).

In [36]:
from weaviate.classes.query import Filter

results_filtered = collection.query.near_vector(
    near_vector=query_vec[0].tolist(),
    limit=5,
    return_metadata=MetadataQuery(distance=True),
    return_properties=["text", "doc_id", "chunk_id"],
    filters=Filter.by_property("doc_id").equal(1)
)

print("Resultados filtrados (doc_id=1):")
for obj in results_filtered.objects:
    print(f"  dist={obj.metadata.distance:.4f}  chunk={obj.properties['chunk_id']}  → {obj.properties['text'][:120]}...")

Resultados filtrados (doc_id=1):
  dist=0.8752  chunk=2  → lectrical code in many jurisdictions. This large mushroom shape can also be found in buttons for use with operators who ...
  dist=0.9818  chunk=4  → ons. Akin to fire alarm switches, some big red buttons, when deployed with suitable visual and audible warnings such as ...
  dist=0.9851  chunk=0  → Push-button A push-button (also spelled pushbutton) or simply button is a simple switch mechanism for controlling some a...
  dist=0.9992  chunk=1  → appliances, and various other mechanical and electronic devices, home and commercial. In industrial and commercial appli...
  dist=0.9996  chunk=3  → art button when pushed will cause the process or machine operation to be started and a secondary contact designed into t...


### Respuestas

- **¿Schema + objetos vs tabla + filas?** El enfoque de schema + objetos es más flexible: cada objeto puede tener propiedades dinámicas y un vector asociado sin necesidad de definir relaciones rígidas. En una tabla SQL, todo es columnas fijas y los vectores son un tipo de dato más.

- **¿Trade-off complejidad vs expresividad?** Weaviate ofrece más expresividad (filtros por propiedades, tipos de datos, vectorizer configurable), pero requiere más setup que soluciones simples como Chroma. Para proyectos complejos vale la pena, para prototipos puede ser excesivo.

## Parte 6 — Vector Store #4: Chroma (prototipado rápido)

### Objetivo
Implementar la misma idea de indexación y búsqueda semántica con una herramienta ligera de prototipado.

### Qué debes implementar
1. Crear una colección.
2. Insertar:
   - ids
   - embeddings
   - documents (texto)
   - metadatas (opcional)
3. Consultar Top-k con `query_embedding`.

### Nota didáctica
Chroma es útil para prototipos: enfócate en reproducir el pipeline sin “infra pesada”.

### Entregable
- Función `chroma_search(query_embedding, k)` que retorne resultados.
- Una consulta con `k=5`.

### Preguntas
- ¿Qué tan fácil fue implementar todo comparado con Qdrant/Milvus?
- ¿Qué limitaciones ves para un sistema en producción?


In [37]:
!pip install chromadb -q

c:\Users\ladol\anaconda3\Lib\site-packages\IPython\utils\_process_win32.py:124: ResourceWarning: unclosed file <_io.BufferedWriter name=5>
  return process_handler(cmd, _system_body)
c:\Users\ladol\anaconda3\Lib\site-packages\IPython\utils\_process_win32.py:124: ResourceWarning: unclosed file <_io.BufferedReader name=6>
  return process_handler(cmd, _system_body)
c:\Users\ladol\anaconda3\Lib\site-packages\IPython\utils\_process_win32.py:124: ResourceWarning: unclosed file <_io.BufferedReader name=7>
  return process_handler(cmd, _system_body)


1. Crear una colección.

In [38]:
import chromadb

cclient = chromadb.Client()

col = cclient.create_collection(name="wiki_chunks", metadata={"hnsw:space": "cosine"})


2. Insertar:
   - ids
   - embeddings
   - documents (texto)
   - metadatas (opcional)

In [39]:
batch_size = 500
for start in range(0, len(chunks_df), batch_size):
    end = min(start + batch_size, len(chunks_df))
    col.add(
        ids=[str(i) for i in range(start, end)],
        embeddings=embeddings[start:end].tolist(),
        documents=chunks_df["text"].iloc[start:end].tolist(),
        metadatas=[{"doc_id": int(chunks_df.iloc[i]["doc_id"]), "chunk_id": int(chunks_df.iloc[i]["chunk_id"])} for i in range(start, end)]
    )

print(f"Insertados: {col.count()}")

Insertados: 3773


3. Consultar Top-k con `query_embedding`.

In [40]:
def chroma_search(query_embedding, k=5):
    results = col.query(
        query_embeddings=query_embedding.tolist(),
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )
    salida = []
    for i in range(len(results["ids"][0])):
        salida.append((
            results["ids"][0][i],
            results["distances"][0][i],
            results["documents"][0][i][:120],
            results["metadatas"][0][i]
        ))
    return salida

for id_, score, texto, meta in chroma_search(query_vec, k=5):
    print(f"  id={id_}  dist={score:.4f}  → {texto}...")

  id=2185  dist=0.6212  → ent on the partial pressure of oxygen in the gas to which the cell is exposed: Linearity is achieved by placing a diffus...
  id=122  dist=0.6268  → r cooled to store or extract energy. However, in a non-phase change encapsulated thermal battery the temperature of the ...
  id=117  dist=0.6415  → Thermal Battery A thermal energy battery is a physical structure used for the purpose of storing and releasing thermal e...
  id=3033  dist=0.6481  → sible without outshining the target. The reticle has markings that match targets of various heights from 0.3Â m to 2.7Â ...
  id=2183  dist=0.6745  → Electro-galvanic oxygen sensor An electro-galvanic fuel cell is an electrochemical device which consumes a fuel to produ...


### Respuestas

- **¿Facilidad vs Qdrant/Milvus?** Chroma es notablemente más simple: sin Docker, sin esquema, sin configuración de índices. Tres líneas bastan para crear una colección e insertar datos. Ideal para prototipos rápidos.

- **¿Limitaciones en producción?** Al correr en memoria no persiste datos por defecto, no soporta replicación ni alta disponibilidad, y el rendimiento se degrada con millones de vectores. Para producción se necesitaría una solución como Qdrant o Milvus.

## Parte 7 — SQL + vectores: PostgreSQL/pgvector (vector search transparente)

### Objetivo
Guardar embeddings en una tabla y ejecutar una consulta SQL de similitud.

### Qué debes implementar
1. Conectar a una base PostgreSQL con `pgvector` habilitado.
2. Crear una tabla (ej. `documents`) con:
   - `id` (PK)
   - `text` (texto)
   - `embedding` (vector(D))
   - metadata (columnas adicionales)
3. Insertar todos los documentos y embeddings.
4. Consultar Top-k por similitud, ordenando por distancia.

### Fórmula conceptual (lo que implementa tu SQL)
Para una consulta `q`, buscas:
$$ argmin_d \in D \; \text{dist}(\vec{q}, \vec{d})$$
donde `dist` puede ser L2 o una variante para cosine (según configuración).

### Entregable
- Función `pgvector_search(query_embedding, k)` que ejecute SQL y devuelva:
  - id, score/distancia, text, metadata

### Preguntas
- ¿Qué tan “explicable” te parece esta aproximación vs las otras?
- ¿Qué ventajas ofrece el mundo SQL (JOIN, filtros, agregaciones)?
- ¿Qué limitaciones esperas en escalabilidad frente a bases vectoriales dedicadas?


docker run -d --name pgvector -p 5432:5432 -e POSTGRES_PASSWORD=postgres pgvector/pgvector:pg16

In [45]:
!pip install psycopg[binary] -q

1. Conectar a una base PostgreSQL con `pgvector` habilitado.
2. Crear una tabla (ej. `documents`) con:
   - `id` (PK)
   - `text` (texto)
   - `embedding` (vector(D))
   - metadata (columnas adicionales)

In [47]:
import psycopg

conn = psycopg.connect("host=localhost port=5433 user=postgres password=postgres dbname=postgres")
conn.autocommit = True
cur = conn.cursor()

cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")

cur.execute("DROP TABLE IF EXISTS documents;")
cur.execute("""
    CREATE TABLE documents (
        id INTEGER PRIMARY KEY,
        text TEXT,
        doc_id INTEGER,
        chunk_id INTEGER,
        embedding vector(384)
    );
""")

print("Tabla creada")

Tabla creada


3. Insertar todos los documentos y embeddings.

In [48]:
for i, row in chunks_df.iterrows():
    vec_str = "[" + ",".join(str(x) for x in embeddings[i]) + "]"
    cur.execute(
        "INSERT INTO documents (id, text, doc_id, chunk_id, embedding) VALUES (%s, %s, %s, %s, %s)",
        (i, row["text"][:500], int(row["doc_id"]), int(row["chunk_id"]), vec_str)
    )

print(f"Insertados")
cur.execute("SELECT count(*) FROM documents;")
print(f"Total: {cur.fetchone()[0]}")

Insertados
Total: 3773


4. Consultar Top-k por similitud, ordenando por distancia.

In [49]:
def pgvector_search(query_embedding, k=5):
    vec_str = "[" + ",".join(str(x) for x in query_embedding[0]) + "]"
    cur.execute("""
        SELECT id, text, doc_id, chunk_id, embedding <=> %s::vector AS dist
        FROM documents
        ORDER BY dist ASC
        LIMIT %s
    """, (vec_str, k))
    salida = []
    for row in cur.fetchall():
        salida.append((row[0], row[4], row[1][:120], {"doc_id": row[2], "chunk_id": row[3]}))
    return salida

for id_, score, texto, meta in pgvector_search(query_vec, k=5):
    print(f"  id={id_}  dist={score:.4f}  → {texto}...")

  id=2185  dist=0.6212  → ent on the partial pressure of oxygen in the gas to which the cell is exposed: Linearity is achieved by placing a diffus...
  id=122  dist=0.6268  → r cooled to store or extract energy. However, in a non-phase change encapsulated thermal battery the temperature of the ...
  id=117  dist=0.6415  → Thermal Battery A thermal energy battery is a physical structure used for the purpose of storing and releasing thermal e...
  id=3033  dist=0.6481  → sible without outshining the target. The reticle has markings that match targets of various heights from 0.3Â m to 2.7Â ...
  id=2183  dist=0.6745  → Electro-galvanic oxygen sensor An electro-galvanic fuel cell is an electrochemical device which consumes a fuel to produ...


### Respuestas

- **¿Qué tan explicable es esta aproximación?** Muy explicable: la query SQL muestra exactamente qué se calcula (distancia coseno con `<=>`), qué se ordena y qué se retorna. Cualquier persona con conocimiento SQL puede entender el flujo sin necesidad de aprender una API nueva.

- **¿Ventajas del mundo SQL?** Se pueden hacer JOINs con otras tablas, filtros complejos con WHERE, agregaciones con GROUP BY, y combinar búsqueda vectorial con lógica relacional en una sola query.

- **¿Limitaciones en escalabilidad?** PostgreSQL no fue diseñado para búsqueda vectorial masiva. Con millones de vectores el rendimiento se degrada frente a soluciones dedicadas como Milvus o Qdrant que implementan índices ANN optimizados.

In [50]:
cur.close()
conn.close()
print("Conexion cerrada")

Conexion cerrada


## Comparativa General

| Motor | Tipo | Metadata | Setup | Mejor para |
|-------|------|----------|-------|------------|
| FAISS | Librería | No nativo | pip | Búsqueda pura, prototipos |
| Qdrant | Base vectorial | Payload nativo | Docker | Producción con filtros |
| Milvus | Base vectorial | Esquema tipado | Docker/Lite | Escalabilidad, índices ANN |
| Weaviate | Base vectorial | Schema + propiedades | Docker | Búsqueda semántica estructurada |
| Chroma | Vector store | Dict simple | Ninguno | Prototipado rápido, RAG |
| pgvector | Extensión SQL | Columnas SQL | Docker (PG) | Integración con sistemas relacionales |

Los 6 motores retornaron los mismos Top-5 para la query "Battery measuring", validando la consistencia entre implementaciones. La diferencia principal está en el trade-off entre simplicidad de setup (Chroma, FAISS) y funcionalidad para producción (Qdrant, Weaviate, Milvus).